In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!apt-get update -qq
!apt-get install -y -qq libxrender1 libxi6 libxkbcommon0 libsm6 libxxf86vm1 libgl1
!wget -q https://download.blender.org/release/Blender4.2/blender-4.2.1-linux-x64.tar.xz -P /tmp
!tar -xf /tmp/blender-4.2.1-linux-x64.tar.xz -C /tmp
BL = '/tmp/blender-4.2.1-linux-x64/blender'
!$BL --version | head -1

In [ ]:
%%writefile /kaggle/working/inspect.py
import bpy
for o in bpy.data.objects:
    print(f'{o.type:8s} | {o.name:40s} | dims={tuple(round(x,2) for x in o.dimensions)}')
print('--- polygons:', sum(len(o.data.polygons) for o in bpy.data.objects if o.type=='MESH'))

In [ ]:
!find /kaggle/input -maxdepth 4 | head -40

In [ ]:
!find /kaggle/input/datasets/utkarshraj23/model1 -maxdepth 4 | head -40

In [ ]:
%%writefile /kaggle/working/inspect.py
import bpy
print('=== OBJECTS ===')
for o in bpy.data.objects:
    print(f'{o.type:8s} | {o.name:40s} | dims={tuple(round(x,2) for x in o.dimensions)}')
print('POLYS', sum(len(o.data.polygons) for o in bpy.data.objects if o.type=='MESH'))
print('=== TEXTURES ===')
for i in bpy.data.images:
    print(f'IMG {i.name:35s} size={i.size[0]}x{i.size[1]} packed={i.packed_file is not None} src={i.source} path={i.filepath}')

In [ ]:
BLEND='/kaggle/input/datasets/utkarshraj23/model1/Blender/Blender_004.blend'
!/tmp/blender-4.2.1-linux-x64/blender $BLEND --background --python /kaggle/working/inspect.py 2>&1 | grep -E "^(MESH|LIGHT|CAMERA|EMPTY|POLYS|IMG|===)"

In [ ]:
%%writefile /kaggle/working/inspect2.py
import bpy, mathutils
for o in sorted([o for o in bpy.data.objects if o.type=='MESH'],
                key=lambda x: -len(x.data.polygons)):
    c = o.matrix_world.translation
    print(f'{o.name:22s} polys={len(o.data.polygons):6d} '
          f'loc=({c.x:8.1f},{c.y:8.1f},{c.z:8.1f}) '
          f'dims=({o.dimensions.x:7.1f},{o.dimensions.y:7.1f},{o.dimensions.z:7.1f}) '
          f'parent={o.parent.name if o.parent else "-"}')

In [ ]:
BLEND='/kaggle/input/datasets/utkarshraj23/model1/Blender/Blender_004.blend'
!/tmp/blender-4.2.1-linux-x64/blender $BLEND --background --python /kaggle/working/inspect2.py 2>&1 | grep -E "polys="

In [ ]:
%%writefile /kaggle/working/inspect3.py
import bpy, mathutils
for o in bpy.data.objects:
    if o.type != 'MESH': continue
    pts = [o.matrix_world @ mathutils.Vector(c) for c in o.bound_box]
    lo = [min(p[i] for p in pts) for i in range(3)]
    hi = [max(p[i] for p in pts) for i in range(3)]
    print(f'{o.name:20s} hide_render={o.hide_render} hide_vp={o.hide_viewport} '
          f'world_z=[{lo[2]:8.1f},{hi[2]:8.1f}] world_x=[{lo[0]:8.1f},{hi[0]:8.1f}]')

In [ ]:
!/tmp/blender-4.2.1-linux-x64/blender $BLEND --background --python /kaggle/working/inspect3.py 2>&1 | grep hide_render

In [ ]:
%%writefile /kaggle/working/testrender.py
import bpy, math, mathutils
KEEP = {'Building_Base', 'Details', 'Glass', 'Building_04'}
for o in list(bpy.data.objects):
    if o.name not in KEEP:
        bpy.data.objects.remove(o, do_unlink=True)
bpy.context.view_layer.update()

pts = [o.matrix_world @ mathutils.Vector(c)
       for o in bpy.data.objects if o.type == 'MESH' for c in o.bound_box]
lo = mathutils.Vector([min(p[i] for p in pts) for i in range(3)])
hi = mathutils.Vector([max(p[i] for p in pts) for i in range(3)])
ctr = (lo + hi) / 2
R = math.hypot(hi.x - lo.x, hi.y - lo.y) * 1.6      # comfortably outside
print(f'BBOX lo={tuple(round(v,1) for v in lo)} hi={tuple(round(v,1) for v in hi)}')
print(f'CENTRE {tuple(round(v,1) for v in ctr)}  CAM_R {R:.1f}')

bpy.ops.object.light_add(type='SUN', location=(ctr.x+80, ctr.y-80, hi.z+60))
bpy.context.object.data.energy = 4
bpy.context.object.rotation_euler = (0.6, 0.1, 0.9)

bpy.ops.object.empty_add(location=ctr)
tgt = bpy.context.object
bpy.ops.object.camera_add(location=(ctr.x + R, ctr.y - R, hi.z * 0.7))
cam = bpy.context.object
cam.data.lens, cam.data.sensor_width = 24.0, 36.0
cam.data.clip_end = 5000                     # <-- likely culprit
bpy.context.scene.camera = cam
c = cam.constraints.new('TRACK_TO'); c.target = tgt
c.track_axis, c.up_axis = 'TRACK_NEGATIVE_Z', 'UP_Y'

sc = bpy.context.scene
sc.render.engine = 'CYCLES'; sc.cycles.samples = 32; sc.cycles.device = 'CPU'
sc.render.resolution_x, sc.render.resolution_y = 1024, 768
sc.render.film_transparent = False
sc.render.filepath = '/kaggle/working/testrender.png'
bpy.ops.render.render(write_still=True)
print('TEST RENDER DONE')

In [ ]:
!/tmp/blender-4.2.1-linux-x64/blender $BLEND --background --python /kaggle/working/testrender.py 2>&1 | tail -5
from PIL import Image; Image.open('/kaggle/working/testrender.png')

In [ ]:
%%writefile /kaggle/working/flight.py
import bpy, bmesh, math, json, os, mathutils

# ================== CONFIG ==================
STRATEGY = 'orbit_single'      # orbit_single | orbit_multi | orbit_high
RUN_ID   = 'run_002'
N_VIEWS  = 24
RADIUS_MULT = 2.8              # orbit radius / structure radius
LENS_MM, SENSOR_MM = 20.0, 36.0
RES_X, RES_Y, SAMPLES = 768, 1024, 32
PREVIEW_ONLY = False            # True -> render 1 image only
KEEP = {'Building_Base', 'Details', 'Glass', 'Building_04'}
# ============================================

OUT = f'/kaggle/working/{RUN_ID}'
os.makedirs(f'{OUT}/images', exist_ok=True)
os.makedirs(f'{OUT}/exports', exist_ok=True)

# ---------- 1. strip the scene to the building ----------
for o in list(bpy.data.objects):
    if o.name not in KEEP:
        bpy.data.objects.remove(o, do_unlink=True)
bpy.context.view_layer.update()

meshes = [o for o in bpy.data.objects if o.type == 'MESH']
pts = [o.matrix_world @ mathutils.Vector(c) for o in meshes for c in o.bound_box]
lo = mathutils.Vector([min(p[i] for p in pts) for i in range(3)])
hi = mathutils.Vector([max(p[i] for p in pts) for i in range(3)])
ctr = (lo + hi) / 2
struct_r = math.hypot(hi.x - lo.x, hi.y - lo.y) / 2
R = struct_r * RADIUS_MULT
Hb = hi.z - lo.z
print(f'SCENE {Hb:.1f} m tall, radius {struct_r:.1f} m -> orbit R={R:.1f} m')

# ---------- 2. flight strategies ----------
def orbit_single(n):
    """One ring at mid-height."""
    return [(ctr.x + R*math.cos(2*math.pi*i/n),
             ctr.y + R*math.sin(2*math.pi*i/n),
             lo.z + Hb*0.50) for i in range(n)]

def orbit_multi(n, rings=3):
    """Three rings, staggered in azimuth so views interleave."""
    per, P = n // rings, []
    for k in range(rings):
        alt = lo.z + Hb * (0.20 + 0.30*k)
        for i in range(per):
            a = 2*math.pi*i/per + k*(2*math.pi/per)/rings
            P.append((ctr.x + R*math.cos(a), ctr.y + R*math.sin(a), alt))
    return P

def orbit_high(n):
    """Single ring above roof level, looking down."""
    return [(ctr.x + R*0.8*math.cos(2*math.pi*i/n),
             ctr.y + R*0.8*math.sin(2*math.pi*i/n),
             hi.z + Hb*0.25) for i in range(n)]

positions = {'orbit_single': orbit_single,
             'orbit_multi':  orbit_multi,
             'orbit_high':   orbit_high}[STRATEGY](N_VIEWS)

# ---------- 3. ground plane, textured so SfM can use it ----------
bpy.ops.mesh.primitive_plane_add(size=R*8, location=(ctr.x, ctr.y, lo.z))
gnd = bpy.context.object
gnd.name = 'GroundPlane'
mat = bpy.data.materials.new('ground'); mat.use_nodes = True
nt = mat.node_tree; bsdf = nt.nodes['Principled BSDF']
nz = nt.nodes.new('ShaderNodeTexNoise')
nz.inputs['Scale'].default_value = 40.0
nz.inputs['Detail'].default_value = 12.0
ramp = nt.nodes.new('ShaderNodeValToRGB')
ramp.color_ramp.elements[0].position = 0.35
ramp.color_ramp.elements[1].position = 0.65
ramp.color_ramp.elements[0].color = (0.05, 0.05, 0.05, 1)
ramp.color_ramp.elements[1].color = (0.35, 0.33, 0.30, 1)
nt.links.new(nz.outputs['Fac'], ramp.inputs['Fac'])
nt.links.new(ramp.outputs['Color'], bsdf.inputs['Base Color'])
bsdf.inputs['Roughness'].default_value = 0.95
gnd.data.materials.append(mat)

# ---------- 4. sun, angled for relief on the facades ----------
bpy.ops.object.light_add(type='SUN', location=(ctr.x + R, ctr.y - R, hi.z + R))
sun = bpy.context.object
sun.data.energy = 3            # was 5
sun.data.angle = 0.1           # was 0.02 — softer shadow edges
sun.rotation_euler = (math.radians(50), 0, math.radians(135))

# world ambient — lifts the shadow side so it still has texture
world = bpy.context.scene.world or bpy.data.worlds.new('World')
bpy.context.scene.world = world
world.use_nodes = True
bg = world.node_tree.nodes['Background']
bg.inputs['Color'].default_value = (0.35, 0.40, 0.48, 1.0)
bg.inputs['Strength'].default_value = 1.2

# ---------- 5. camera aimed at the building's centre ----------
bpy.ops.object.empty_add(location=ctr)
tgt = bpy.context.object
bpy.ops.object.camera_add()
cam = bpy.context.object
cam.data.lens, cam.data.sensor_width = LENS_MM, SENSOR_MM
cam.data.clip_end = 5000
bpy.context.scene.camera = cam
con = cam.constraints.new('TRACK_TO')
con.target, con.track_axis, con.up_axis = tgt, 'TRACK_NEGATIVE_Z', 'UP_Y'

sc = bpy.context.scene
sc.render.engine = 'CYCLES'
sc.cycles.samples = SAMPLES
sc.cycles.device = 'CPU'
sc.render.resolution_x, sc.render.resolution_y = RES_X, RES_Y

# ---------- 6. fly ----------
poses = []
todo = positions[:1] if PREVIEW_ONLY else positions
for i, p in enumerate(todo):
    cam.location = p
    bpy.context.view_layer.update()
    name = f'img_{i:03d}.png'
    sc.render.filepath = f'{OUT}/images/{name}'
    bpy.ops.render.render(write_still=True)
    poses.append({'file': name,
                  'location': list(cam.matrix_world.translation),
                  'matrix_world': [list(r) for r in cam.matrix_world]})
    print(f'RENDERED {i+1}/{len(todo)}')

# ---------- 7. ground-truth mesh (building only, no ground) ----------
verts, faces = [], []
for o in bpy.data.objects:
    if o.type != 'MESH' or o.name == 'GroundPlane':
        continue
    bm = bmesh.new(); bm.from_mesh(o.data)
    bm.transform(o.matrix_world)
    bmesh.ops.triangulate(bm, faces=bm.faces)
    bm.verts.index_update()
    base = len(verts)
    verts += [tuple(v.co) for v in bm.verts]
    faces += [tuple(base + v.index for v in f.verts) for f in bm.faces]
    bm.free()
with open(f'{OUT}/exports/gt_mesh.ply', 'w') as f:
    f.write(f'ply\nformat ascii 1.0\nelement vertex {len(verts)}\n'
            'property float x\nproperty float y\nproperty float z\n'
            f'element face {len(faces)}\n'
            'property list uchar int vertex_indices\nend_header\n')
    for v in verts: f.write(f'{v[0]:.6f} {v[1]:.6f} {v[2]:.6f}\n')
    for t in faces: f.write(f'3 {t[0]} {t[1]} {t[2]}\n')
print(f'GT MESH {len(verts)} verts {len(faces)} tris')

# ---------- 8. flight record ----------
if not PREVIEW_ONLY:
    P = [list(p) for p in positions]
    seg = [math.dist(P[i], P[(i+1) % len(P)]) for i in range(len(P))]
    SPEED, DWELL = 3.0, 2.0
    json.dump({'run': RUN_ID, 'strategy': STRATEGY,
               'params': dict(N_VIEWS=N_VIEWS, RADIUS=R, RADIUS_MULT=RADIUS_MULT,
                              LENS_MM=LENS_MM, SENSOR_MM=SENSOR_MM,
                              RES_X=RES_X, RES_Y=RES_Y, SAMPLES=SAMPLES,
                              bbox_lo=list(lo), bbox_hi=list(hi)),
               'speed_mps': SPEED, 'dwell_s': DWELL,
               'path_length_m': sum(seg),
               'flight_time_s': sum(seg)/SPEED + DWELL*len(P),
               'poses': poses},
              open(f'{OUT}/exports/flight.json', 'w'), indent=2)
    print(f'PATH {sum(seg):.1f} m')
print('SCRIPT DONE')

In [ ]:
!/tmp/blender-4.2.1-linux-x64/blender $BLEND --background --python /kaggle/working/flight.py 2>&1 | grep -E "SCENE|RENDERED|GT MESH|PATH|DONE|Error|Traceback"
from PIL import Image; Image.open('/kaggle/working/run_002/images/img_000.png')

In [ ]:
!du -sh /kaggle/working/*
!ls /kaggle/working/run_002/images | wc -l

In [ ]:
!ls /kaggle/working/run_002/images | wc -l

In [ ]:
!ls -la /kaggle/working/run_002/exports/